## Init Spark application

In [1]:
import pyspark
from delta import *

builder = (
    pyspark.sql.SparkSession.builder.appName("delta_lake_tutorial")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config(
        "spark.sql.catalog.spark_catalog",
        "org.apache.spark.sql.delta.catalog.DeltaCatalog",
    )    
    .config("spark.sql.warehouse.dir", "../spark-warehouse")    
    # .config("spark.driver.extraJavaOptions", "-Dderby.system.home=../")        
    # .enableHiveSupport()
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()

25/06/05 16:35:52 WARN Utils: Your hostname, 0726StevenFanChiang.local resolves to a loopback address: 127.0.0.1; using 192.168.100.203 instead (on interface en0)
25/06/05 16:35:52 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Ivy Default Cache set to: /Users/steven.fanchiang/.ivy2/cache
The jars for the packages stored in: /Users/steven.fanchiang/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-0b9cc848-bea7-46e9-a8bc-636bd6d95205;1.0
	confs: [default]


:: loading settings :: url = jar:file:/Users/steven.fanchiang/miniconda3/envs/spark-cluster/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


	found io.delta#delta-spark_2.12;3.3.1 in central
	found io.delta#delta-storage;3.3.1 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 123ms :: artifacts dl 7ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.3.1 from central in [default]
	io.delta#delta-storage;3.3.1 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |   0   |   0   |   0   ||   3   |   0   |
	---------------------------------------------------------------------
:: retrieving :: org.apache.spark#spark-submit-parent-0b9cc848-bea7-46e9-a8bc-636bd6d95205
	confs: [default]
	0 artifacts copied, 3 already retrieved (0kB/3ms)
25/06/05 16:35:52

## Create delta table

In [2]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, TimestampType

schema = StructType([
  StructField("id", IntegerType(), True),
  StructField("firstName", StringType(), True),
  StructField("middleName", StringType(), True),
  StructField("lastName", StringType(), True),
  StructField("gender", StringType(), True),
  StructField("birthDate", TimestampType(), True),
  StructField("ssn", StringType(), True),
  StructField("salary", IntegerType(), True)
])

df = spark.read.format("csv").option("header", True).schema(schema).load("../data/people_10m.csv")


# If you know the table does not already exist, you can call this instead:
df.write.format("delta").mode("Overwrite").saveAsTable("people_10m")


# Create the table if it does not exist. Otherwise, replace the existing table.
# df.writeTo("people_10m").createOrReplace()


# SQL version
# CREATE OR REPLACE TABLE main.default.people_10m (
#   id INT,
#   firstName STRING,
#   middleName STRING,
#   lastName STRING,
#   gender STRING,
#   birthDate TIMESTAMP,
#   ssn STRING,
#   salary INT
# );

# COPY INTO main.default.people_10m
# FROM '/Volumes/main/default/my-volume/export.csv'
# FILEFORMAT = CSV
# FORMAT_OPTIONS ( 'header' = 'true', 'inferSchema' = 'true' );

25/06/05 16:36:19 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

## Describe created table

In [3]:
spark.sql("SHOW tables").show()

spark.sql("describe table EXTENDED people_10m").show(100, False)

# spark.sql("SHOW catalogs").show()
# spark.catalog.listTables()

+---------+----------+-----------+
|namespace| tableName|isTemporary|
+---------+----------+-----------+
|  default|people_10m|      false|
+---------+----------+-----------+

+----------------------------+---------------------------------------------------------------------------------------------------------------------+-------+
|col_name                    |data_type                                                                                                            |comment|
+----------------------------+---------------------------------------------------------------------------------------------------------------------+-------+
|id                          |int                                                                                                                  |NULL   |
|firstName                   |string                                                                                                               |NULL   |
|middleName                  |string   

## Read a table
You access data in Delta tables by the table name or the table path, as shown in the following examples:

In [4]:
people_df = spark.read.table("people_10m")

people_df.show()
people_df.count()

+---+----------+----------+-------------+------+-------------------+-----------+------+
| id| firstName|middleName|     lastName|gender|          birthDate|        ssn|salary|
+---+----------+----------+-------------+------+-------------------+-----------+------+
|  1|    Pennie|     Carry|   Hirschmann|     F|1955-07-02 13:00:00|981-43-9345| 56172|
|  2|        An|     Amira|       Cowper|     F|1992-02-08 13:00:00|978-97-8086| 40203|
|  3|     Quyen|    Marlen|         Dome|     F|1970-10-11 12:00:00|957-57-8246| 53417|
|  4|   Coralie|  Antonina|      Marshal|     F|1990-04-11 12:00:00|963-39-4885| 94727|
|  5|    Terrie|      Wava|        Bonar|     F|1980-01-16 13:00:00|964-49-8051| 79908|
|  6|  Chassidy|Concepcion|Bourthouloume|     F|1990-11-24 13:00:00|954-59-9172| 64652|
|  7|      Geri|    Tambra|        Mosby|     F|1970-12-19 13:00:00|968-16-4020| 38195|
|  8|    Patria|     Nancy|      Arstall|     F|1985-01-02 13:00:00|984-76-3770|102053|
|  9|    Terese|  Alfredia|     

1000

## Update a table
You can update data that matches a predicate in a Delta table. For example, in the example people_10m table, to change an abbreviation in the gender column from M or F to Male or Female, you can run the following:

In [5]:
from delta.tables import *
from pyspark.sql.functions import *

deltaTable = DeltaTable.forName(spark, "people_10m")

# Declare the predicate by using a SQL-formatted string.
deltaTable.update(
  condition = "gender = 'F'",
  set = { "gender": "'Female'" }
)

# Declare the predicate by using Spark SQL functions.
deltaTable.update(
  condition = col('gender') == 'M',
  set = { 'gender': lit('Male') }
)

## Read table again

In [6]:
people_df = spark.read.table("people_10m")

people_df.show()

+---+----------+----------+-------------+------+-------------------+-----------+------+
| id| firstName|middleName|     lastName|gender|          birthDate|        ssn|salary|
+---+----------+----------+-------------+------+-------------------+-----------+------+
|  1|    Pennie|     Carry|   Hirschmann|Female|1955-07-02 13:00:00|981-43-9345| 56172|
|  2|        An|     Amira|       Cowper|Female|1992-02-08 13:00:00|978-97-8086| 40203|
|  3|     Quyen|    Marlen|         Dome|Female|1970-10-11 12:00:00|957-57-8246| 53417|
|  4|   Coralie|  Antonina|      Marshal|Female|1990-04-11 12:00:00|963-39-4885| 94727|
|  5|    Terrie|      Wava|        Bonar|Female|1980-01-16 13:00:00|964-49-8051| 79908|
|  6|  Chassidy|Concepcion|Bourthouloume|Female|1990-11-24 13:00:00|954-59-9172| 64652|
|  7|      Geri|    Tambra|        Mosby|Female|1970-12-19 13:00:00|968-16-4020| 38195|
|  8|    Patria|     Nancy|      Arstall|Female|1985-01-02 13:00:00|984-76-3770|102053|
|  9|    Terese|  Alfredia|     

## Merge table
To merge a set of updates and insertions into an existing Delta table, you use the DeltaTable.merge method for Python and Scala, and the MERGE INTO statement for SQL.

In [7]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType
from datetime import date

schema = StructType([
  StructField("id", IntegerType(), True),
  StructField("firstName", StringType(), True),
  StructField("middleName", StringType(), True),
  StructField("lastName", StringType(), True),
  StructField("gender", StringType(), True),
  StructField("birthDate", DateType(), True),
  StructField("ssn", StringType(), True),
  StructField("salary", IntegerType(), True)
])

data = [
  (9999998, 'Billy', 'Tommie', 'Luppitt', 'Male', date.fromisoformat('1992-09-17'), '953-38-9452', 55250),
  (9999999, 'Elias', 'Cyril', 'Leadbetter', 'Male', date.fromisoformat('1984-05-22'), '906-51-2137', 48500),
  (10000000, 'Joshua', 'Chas', 'Broggio', 'Male', date.fromisoformat('1968-07-22'), '988-61-6247', 90000),
  (20000001, 'John', '', 'Doe', 'Male', date.fromisoformat('1978-01-14'), '345-67-8901', 55500),
  (20000002, 'Mary', '', 'Smith', 'Female', date.fromisoformat('1982-10-29'), '456-78-9012', 98250),
  (20000003, 'Jane', '', 'Doe', 'Female', date.fromisoformat('1981-06-25'), '567-89-0123', 89900)
]

people_10m_updates = spark.createDataFrame(data, schema)
people_10m_updates.createTempView("people_10m_updates")

# ...

from delta.tables import DeltaTable

deltaTable = DeltaTable.forName(spark, 'people_10m')

(deltaTable.alias("people_10m")
  .merge(
    people_10m_updates.alias("people_10m_updates"),
    "people_10m.id = people_10m_updates.id")
  .whenMatchedUpdateAll()
  .whenNotMatchedInsertAll()
  .execute()
)


# SQL version
# """
# CREATE OR REPLACE TEMP VIEW people_10m_updates (
#   id, firstName, middleName, lastName, gender, birthDate, ssn, salary
# ) AS VALUES
#   (9999998, 'Billy', 'Tommie', 'Luppitt', 'M', '1992-09-17T04:00:00.000+0000', '953-38-9452', 55250),
#   (9999999, 'Elias', 'Cyril', 'Leadbetter', 'M', '1984-05-22T04:00:00.000+0000', '906-51-2137', 48500),
#   (10000000, 'Joshua', 'Chas', 'Broggio', 'M', '1968-07-22T04:00:00.000+0000', '988-61-6247', 90000),
#   (20000001, 'John', '', 'Doe', 'M', '1978-01-14T04:00:00.000+000', '345-67-8901', 55500),
#   (20000002, 'Mary', '', 'Smith', 'F', '1982-10-29T01:00:00.000+000', '456-78-9012', 98250),
#   (20000003, 'Jane', '', 'Doe', 'F', '1981-06-25T04:00:00.000+000', '567-89-0123', 89900);

# MERGE INTO people_10m
# USING people_10m_updates
# ON people_10m.id = people_10m_updates.id
# WHEN MATCHED THEN UPDATE SET *
# WHEN NOT MATCHED THEN INSERT *;
# """


## Query with conditions

In [8]:
df = spark.read.table("people_10m")
df_filtered = df.filter(df["id"] >= 9999998)
df_filtered.show()

df_filtered.count()

# SQL version
# """
# SELECT * FROM main.default.people_10m WHERE id >= 9999998
# """

+--------+---------+----------+----------+------+-------------------+-----------+------+
|      id|firstName|middleName|  lastName|gender|          birthDate|        ssn|salary|
+--------+---------+----------+----------+------+-------------------+-----------+------+
| 9999999|    Elias|     Cyril|Leadbetter|  Male|1984-05-22 00:00:00|906-51-2137| 48500|
| 9999998|    Billy|    Tommie|   Luppitt|  Male|1992-09-17 00:00:00|953-38-9452| 55250|
|10000000|   Joshua|      Chas|   Broggio|  Male|1968-07-22 00:00:00|988-61-6247| 90000|
|20000002|     Mary|          |     Smith|Female|1982-10-29 00:00:00|456-78-9012| 98250|
|20000003|     Jane|          |       Doe|Female|1981-06-25 00:00:00|567-89-0123| 89900|
|20000001|     John|          |       Doe|  Male|1978-01-14 00:00:00|345-67-8901| 55500|
+--------+---------+----------+----------+------+-------------------+-----------+------+



6

## Delete from a table
You can remove data that matches a predicate from a Delta table. For instance, in the example people_10m table, to delete all rows corresponding to people with a value in the birthDate column from before 1955, you can run the following:

In [9]:
from delta.tables import *
from pyspark.sql.functions import *

deltaTable = DeltaTable.forName(spark, "people_10m")

# Declare the predicate by using a SQL-formatted string.
deltaTable.delete("birthDate < '1955-01-01'")

# Declare the predicate by using Spark SQL functions.
deltaTable.delete(col('birthDate') < '1960-01-01')

# SQL version
# DELETE FROM main.default.people_10m WHERE birthDate < '1955-01-01'

## Read table again

In [10]:
df = spark.read.table("people_10m")
cnt = df.count()
print(f"the data count: {cnt}")

the data count: 839


## Query an earlier version of the table (time travel)

Delta Lake time travel allows you to query an older snapshot of a Delta table.

In [12]:
from delta.tables import *

deltaTable = DeltaTable.forName(spark, "people_10m")
deltaHistory = deltaTable.history()

deltaHistory.show()
deltaHistory.printSchema()

+-------+--------------------+------+--------+--------------------+--------------------+----+--------+---------+-----------+--------------+-------------+--------------------+------------+--------------------+
|version|           timestamp|userId|userName|           operation| operationParameters| job|notebook|clusterId|readVersion|isolationLevel|isBlindAppend|    operationMetrics|userMetadata|          engineInfo|
+-------+--------------------+------+--------+--------------------+--------------------+----+--------+---------+-----------+--------------+-------------+--------------------+------------+--------------------+
|      4|2025-06-05 16:40:...|  NULL|    NULL|              DELETE|{predicate -> ["(...|NULL|    NULL|     NULL|          3|  Serializable|        false|{numRemovedFiles ...|        NULL|Apache-Spark/3.5....|
|      3|2025-06-05 16:40:...|  NULL|    NULL|              DELETE|{predicate -> ["(...|NULL|    NULL|     NULL|          2|  Serializable|        false|{numRemoved

To query an older version of a table, specify the table's version or timestamp. For example, to query version 0 or timestamp 2024-05-15T22:43:15.000+00:00Z from the preceding history, use the following:

In [13]:
deltaHistory.where("version == 0").show()
# Or:
deltaHistory.where("timestamp == '2024-05-15T22:43:15.000+00:00'").show()

+-------+--------------------+------+--------+--------------------+--------------------+----+--------+---------+-----------+--------------+-------------+--------------------+------------+--------------------+
|version|           timestamp|userId|userName|           operation| operationParameters| job|notebook|clusterId|readVersion|isolationLevel|isBlindAppend|    operationMetrics|userMetadata|          engineInfo|
+-------+--------------------+------+--------+--------------------+--------------------+----+--------+---------+-----------+--------------+-------------+--------------------+------------+--------------------+
|      0|2025-06-05 16:36:...|  NULL|    NULL|CREATE OR REPLACE...|{partitionBy -> [...|NULL|    NULL|     NULL|       NULL|  Serializable|        false|{numFiles -> 1, n...|        NULL|Apache-Spark/3.5....|
+-------+--------------------+------+--------+--------------------+--------------------+----+--------+---------+-----------+--------------+-------------+-----------

## Query older version
DataFrameReader options allow you to create a DataFrame from a Delta table that is fixed to a specific version or timestamp of the table, for example:

In [14]:
from delta.tables import *

deltaTable = DeltaTable.forName(spark, "people_10m")
deltaHistory = deltaTable.history()


df = spark.read.option('versionAsOf', 1).table("people_10m")
# # Or:
# df = spark.read.option('timestampAsOf', '2024-05-15T22:43:15.000+00:00').table("people_10m")

df.show()
df.count()

+---+----------+----------+-------------+------+-------------------+-----------+------+
| id| firstName|middleName|     lastName|gender|          birthDate|        ssn|salary|
+---+----------+----------+-------------+------+-------------------+-----------+------+
|  1|    Pennie|     Carry|   Hirschmann|Female|1955-07-02 13:00:00|981-43-9345| 56172|
|  2|        An|     Amira|       Cowper|Female|1992-02-08 13:00:00|978-97-8086| 40203|
|  3|     Quyen|    Marlen|         Dome|Female|1970-10-11 12:00:00|957-57-8246| 53417|
|  4|   Coralie|  Antonina|      Marshal|Female|1990-04-11 12:00:00|963-39-4885| 94727|
|  5|    Terrie|      Wava|        Bonar|Female|1980-01-16 13:00:00|964-49-8051| 79908|
|  6|  Chassidy|Concepcion|Bourthouloume|Female|1990-11-24 13:00:00|954-59-9172| 64652|
|  7|      Geri|    Tambra|        Mosby|Female|1970-12-19 13:00:00|968-16-4020| 38195|
|  8|    Patria|     Nancy|      Arstall|Female|1985-01-02 13:00:00|984-76-3770|102053|
|  9|    Terese|  Alfredia|     

1000

## Save older version as new table

In [15]:
df = spark.read.option('versionAsOf', 1).table("people_10m")
df.write.format("delta").mode("Overwrite").saveAsTable("people_10m_bronze")

## Clean up snapshots with VACUUM
Delta Lake provides snapshot isolation for reads, which means that it is safe to run an optimize operation even while other users or jobs are querying the table. Eventually however, you should clean up old snapshots. You can do this by running the vacuum operation:

In [3]:
from delta.tables import *

deltaTable = DeltaTable.forName(spark, "people_10m")
deltaTable.vacuum()

25/06/02 09:56:15 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

Deleted 0 files and directories in a total of 1 directories.


DataFrame[]